

# Part 1: Exact Architecture Breakdown with Layer-by-Layer Weight Counts

---

## Overview Numbers



In [ ]:
Total parameters in model: Let me compute exactly.



### Component 1: SpectrumEncoder



In [ ]:
Input: spectrum (batch, 1000)
Output: condition embedding (batch, 128)

Layer                  | Shape         | Weights      | Biases  | Total
-----------------------|---------------|------------- |---------|--------
Linear_1               | 1000 → 256    | 256,000      | 256     | 256,256
LeakyReLU(0.01)        | —             | 0            | 0       | 0
Linear_2               | 256 → 256     | 65,536       | 256     | 65,792
LeakyReLU(0.01)        | —             | 0            | 0       | 0
Linear_3               | 256 → 128     | 32,768       | 128     | 32,896
-----------------------|---------------|------------- |---------|--------
SUBTOTAL               |               |              |         | 354,944



### Component 2: Single Coupling Block SubNetwork

Each `CondAffineCoupling` has an internal subnet:



In [ ]:
Input: [x1 (10) ; cond (128)] = 138
Output: [s (10) ; t (10)] = 20

Layer                  | Shape         | Weights      | Biases  | Total
-----------------------|---------------|------------- |---------|--------
Linear_1               | 138 → 256     | 35,328       | 256     | 35,584
LeakyReLU(0.01)        | —             | 0            | 0       | 0
Linear_2               | 256 → 256     | 65,536       | 256     | 65,792
LeakyReLU(0.01)        | —             | 0            | 0       | 0
Linear_3               | 256 → 20      | 5,120        | 20      | 5,140
-----------------------|---------------|------------- |---------|--------
SUBTOTAL (per block)   |               |              |         | 106,516



### Component 3: 8 Coupling Blocks



In [ ]:
8 × 106,516 = 852,128



### Grand Total



In [ ]:
SpectrumEncoder:      354,944
8 Coupling Blocks:    852,128
================================
TOTAL PARAMETERS:   1,207,072  (~1.2M)



---

## Full Data Flow: ASCII Diagram

### TRAINING (Forward Pass)



In [ ]:
┌─────────────────────────────────────────────────────────────────────────────┐
│                         FORWARD PASS (Training)                             │
│                                                                             │
│  x_params (batch, 20)          y_spectrum (batch, 1000)                     │
│      │                              │                                       │
│      │                              ▼                                       │
│      │                     ┌──────────────────┐                             │
│      │                     │  SpectrumEncoder  │                            │
│      │                     │  1000→256→256→128 │                            │
│      │                     └────────┬─────────┘                             │
│      │                              │                                       │
│      │                         cond (batch, 128)                            │
│      │                              │                                       │
│      │    ┌─────────────────────────┼──────────────────────────────────┐    │
│      │    │                         │  REPEATED 8 TIMES               │    │
│      ▼    │                         ▼                                  │    │
│  ┌────────┴───────────────────────────────────────────────────────┐   │    │
│  │  COUPLING BLOCK i                                              │   │    │
│  │                                                                │   │    │
│  │  x (batch, 20)                                                 │   │    │
│  │    │                                                           │   │    │
│  │    ├── x1 = x[:, :10]  (first 10 dims, UNCHANGED)             │   │    │
│  │    │       │                                                   │   │    │
│  │    │       ▼                                                   │   │    │
│  │    │   cat([x1, cond]) → (batch, 138)                          │   │    │
│  │    │       │                                                   │   │    │
│  │    │       ▼                                                   │   │    │
│  │    │   ┌───────────────────────┐                               │   │    │
│  │    │   │  SubNet               │                               │   │    │
│  │    │   │  138 → 256 → 256 → 20│                               │   │    │
│  │    │   └───────────┬───────────┘                               │   │    │
│  │    │               │                                           │   │    │
│  │    │          st (batch, 20)                                   │   │    │
│  │    │           │        │                                      │   │    │
│  │    │      s=st[:,:10]  t=st[:,10:]                             │   │    │
│  │    │      (batch,10)   (batch,10)                              │   │    │
│  │    │           │                                               │   │    │
│  │    │      s = clamp(s, -3, 3)                                  │   │    │
│  │    │                                                           │   │    │
│  │    └── x2 = x[:, 10:]  (last 10 dims, TRANSFORMED)            │   │    │
│  │            │                                                   │   │    │
│  │            ▼                                                   │   │    │
│  │        y2 = x2 * exp(s) + t                                   │   │    │
│  │                                                                │   │    │
│  │    output = cat([x1, y2]) → (batch, 20)                        │   │    │
│  │    log_det += sum(s, dim=1)                                    │   │    │
│  │                                                                │   │    │
│  └────────────────────────────────────────────────────────────────┘   │    │
│           │                                                           │    │
│           ▼                                                           │    │
│      FLIP: x = x[:, [10..19, 0..9]]  (swap halves)                   │    │
│           │                                                           │    │
│           └───── repeat ──────────────────────────────────────────────┘    │
│                                                                             │
│      z (batch, 20)          total_log_det_J (batch,)                        │
│           │                        │                                        │
│           ▼                        ▼                                        │
│      ┌─────────────────────────────────────┐                                │
│      │  NLL Loss                           │                                │
│      │  L = mean(0.5 * sum(z²) - log_det_J)│                                │
│      └─────────────────────────────────────┘                                │
│                    │                                                        │
│                    ▼                                                        │
│              backpropagate                                                  │
└─────────────────────────────────────────────────────────────────────────────┘



### Which Dimensions Get Transformed in Each Block

This is the critical detail. Let me trace which of the 20 parameter indices get transformed at each block:



In [ ]:
ORIGINAL PARAMETER ORDERING:
Index: [0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19]
       [──────── d1-d10 ────────────] [── cavity diams ──] [ρ  η  E  ν ]

BLOCK 1:
  x1 = indices [0..9]   → PASS THROUGH (identity), used to compute s,t
  x2 = indices [10..19] → AFFINE TRANSFORMED: x2*exp(s)+t
  After: FLIP → new order is [10..19, 0..9]

BLOCK 2:
  x1 = indices [10..19] (from prev output) → PASS THROUGH, used to compute s,t
  x2 = indices [0..9]   (from prev output) → AFFINE TRANSFORMED
  After: FLIP → new order is [0..9, 10..19]

BLOCK 3:
  x1 = indices [0..9]   → PASS THROUGH
  x2 = indices [10..19] → AFFINE TRANSFORMED
  After: FLIP

BLOCK 4:
  x1 = indices [10..19] → PASS THROUGH
  x2 = indices [0..9]   → AFFINE TRANSFORMED
  After: FLIP

...pattern continues alternating...

BLOCK 5: transforms [10..19]
BLOCK 6: transforms [0..9]
BLOCK 7: transforms [10..19]
BLOCK 8: transforms [0..9]

SUMMARY:
  Dims [0..9]   transformed in blocks: 2, 4, 6, 8  (4 times)
  Dims [10..19] transformed in blocks: 1, 3, 5, 7  (4 times)
  ✅ SYMMETRIC — each half gets equal number of transformations



### INFERENCE (Reverse Pass)



In [ ]:
┌──────────────────────────────────────────────────────────────────────────────┐
│                        REVERSE PASS (Inference)                              │
│                                                                              │
│   z ~ N(0, I) (batch, 20)        target_spectrum (batch, 1000)               │
│       │                                │                                     │
│       │                                ▼                                     │
│       │                       ┌──────────────────┐                           │
│       │                       │  SpectrumEncoder  │  (SAME weights as        │
│       │                       │  1000→256→256→128 │   training, frozen)      │
│       │                       └────────┬─────────┘                           │
│       │                                │                                     │
│       │                           cond (batch, 128)                          │
│       │                                │                                     │
│       │    ┌───────────────────────────┼─────────────────────────────────┐   │
│       │    │                           │  REPEATED 8 TIMES (REVERSED)   │   │
│       ▼    │                           ▼                                 │   │
│  ┌─────────┴─────────────────────────────────────────────────────────┐  │   │
│  │  STEP 1: UN-FLIP first                                           │  │   │
│  │    x = x[:, [10..19, 0..9]]                                      │  │   │
│  │                                                                   │  │   │
│  │  STEP 2: REVERSE COUPLING BLOCK i                                 │  │   │
│  │    x1 = x[:, :10]   (unchanged half)                              │  │   │
│  │    cat([x1, cond]) → SubNet → s, t                                │  │   │
│  │    x2 = x[:, 10:]                                                 │  │   │
│  │    x2_recovered = (x2 - t) * exp(-s)    ← INVERSE of forward     │  │   │
│  │    output = cat([x1, x2_recovered])                               │  │   │
│  └───────────────────────────────────────────────────────────────────┘  │   │
│           │                                                             │   │
│           └───── repeat (blocks 8→7→6→5→4→3→2→1) ──────────────────────┘   │
│                                                                              │
│       x_scaled (batch, 20)                                                   │
│           │                                                                  │
│           ▼                                                                  │
│   scaler_params.inverse_transform(x_scaled)                                 │
│           │                                                                  │
│           ▼                                                                  │
│   validate_and_clip_parameters(x_physical)                                   │
│           │                                                                  │
│           ▼                                                                  │
│   x_physical (batch, 20)  ← clipped to valid ranges                         │
│           │                                                                  │
│           ▼                                                                  │
│   ┌───────────────────┐                                                      │
│   │  TMM Physics Engine│                                                     │
│   │  (Forward problem) │                                                     │
│   └─────────┬─────────┘                                                      │
│             │                                                                │
│             ▼                                                                │
│   predicted_spectrum (batch, 1000)                                           │
│             │                                                                │
│             ▼                                                                │
│   RMSE vs target → select best candidate                                     │
└──────────────────────────────────────────────────────────────────────────────┘



---

### Weight Update Flow During Training



In [ ]:
                    NLL Loss
                       │
                       ▼
                 ∂L/∂z, ∂L/∂log_det
                       │
         ┌─────────────┴──────────────┐
         │                            │
         ▼                            ▼
   Gradients flow through        Gradients flow through
   8 Coupling Block SubNets      SpectrumEncoder
         │                            │
         ▼                            ▼
   ┌─────────────┐            ┌──────────────────┐
   │  SubNet_1   │            │  Encoder Linear_1│
   │  W: 138→256 │ ← updated │  W: 1000→256     │ ← updated
   │  W: 256→256 │ ← updated │  W: 256→256      │ ← updated
   │  W: 256→20  │ ← updated │  W: 256→128      │ ← updated
   ├─────────────┤            └──────────────────┘
   │  SubNet_2   │
   │  (same arch)│ ← updated
   ├─────────────┤
   │  ...        │
   ├─────────────┤
   │  SubNet_8   │
   │  (same arch)│ ← updated
   └─────────────┘

   ALL weights updated jointly via Adam optimizer.
   No weights are frozen during training.
   Gradient clipping (max_norm=1.0) applied BEFORE optimizer step.



**Key point**: The coupling blocks themselves have **no learnable parameters of their own**. The operations `x2 * exp(s) + t` and `flip` are parameter-free. All learning happens inside the **SubNets** (which output `s` and `t`) and the **SpectrumEncoder** (which outputs `cond`).

---

## Verification: Are 8 Blocks Enough?

The expressivity of a coupling-based flow depends on:

| Factor | Your Model | Typical Range | Assessment |
|--------|-----------|---------------|------------|
| Number of blocks | 8 | 4–16 | ✅ Adequate |
| Transforms per half | 4 each | 2–8 | ✅ Sufficient |
| SubNet depth | 3 layers | 2–5 | ✅ Standard |
| SubNet width | 256 | 128–512 | ✅ Standard |
| Param dim | 20 | — | Low, makes 8 blocks plenty |
| Condition dim | 128 | 64–256 | ✅ Standard |

**For 20-dimensional parameter spaces, 8 affine coupling blocks is well within the accepted range.** The rule of thumb is you need at least `2 × ceil(dim/half_dim) = 4` blocks minimum so each half is transformed at least twice. You have 4 transforms per half, which is good.

---

# Part 2: How to Add Physics-Informed Training Loss

## The Idea

Right now, the loss only measures how "Gaussian" the latent space is:



In [ ]:
L_current = NLL(z, log_det_J) = 0.5 * ||z||² - log|det(J)|



This doesn't directly enforce that **generated parameters produce good spectra**. A physics loss adds:



In [ ]:
L_total = L_NLL + λ * L_physics



where `L_physics` measures the discrepancy between the target spectrum and the spectrum obtained by running the generated parameters through the TMM.

## Implementation

### Step 1: Make TMM Differentiable

Your GPU TMM (`calculate_absorption_full_curve_gpu`) uses raw torch operations—it's **already differentiable**! The complex arithmetic, matrix multiplications, and abs operations all have PyTorch autograd support. You just need to make sure no `.detach()` or `.numpy()` calls break the gradient chain.

**However**, there's a subtle issue: `validate_and_clip_parameters` uses `torch.clamp`, which has zero gradient when values are at the boundary. Use a **soft clamp** instead:



In [ ]:
def soft_clamp(x, low, high, sharpness=10.0):
    """Differentiable alternative to torch.clamp"""
    return low + (high - low) * torch.sigmoid(sharpness * (x - low) / (high - low + 1e-8))



### Step 2: Create a Differentiable Physics Forward Pass



In [ ]:
def differentiable_tmm(params_scaled, scaler_params, device):
    """
    Takes SCALED parameters (output of cINN reverse), 
    converts to physical units, runs TMM, returns absorption spectrum.
    All operations are differentiable.
    """
    # Inverse-scale: convert from StandardScaler space to physical units
    mean = torch.tensor(scaler_params.mean_, dtype=torch.float32, device=device)
    std = torch.tensor(scaler_params.scale_, dtype=torch.float32, device=device)
    params_physical = params_scaled * std + mean
    
    # Soft-clamp to physical bounds instead of hard clamp
    bounds_low = torch.tensor([1e-3]*10 + [20e-3]*6 + [1000, 0.1, 1e7, 0.4], 
                               dtype=torch.float32, device=device)
    bounds_high = torch.tensor([20e-3]*10 + [1980e-3]*6 + [1500, 0.8, 1e8, 0.49],
                                dtype=torch.float32, device=device)
    params_physical = soft_clamp(params_physical, bounds_low, bounds_high)
    
    # Dynamic E constraint (soft version)
    eta = params_physical[:, 17:18]
    E_min = 1e7 * (1 + eta)
    E_max = 1e8 * (1 + eta)
    E_val = params_physical[:, 18:19]
    E_clamped = E_min + (E_max - E_min) * torch.sigmoid(10.0 * (E_val - E_min) / (E_max - E_min + 1e-8))
    params_physical = torch.cat([
        params_physical[:, :18],
        E_clamped,
        params_physical[:, 19:]
    ], dim=1)
    
    # Now run the existing GPU TMM — it's already torch-based
    absorption = calculate_absorption_full_curve_gpu(params_physical, device)
    
    return absorption  # (batch, 1000), differentiable



### Step 3: Modified Training Loop



In [ ]:
def combined_loss(model, x_params, y_spectra, scaler_params, device, 
                  lambda_physics=0.1, physics_subsample=64):
    """
    Combined NLL + Physics loss.
    
    Args:
        x_params: scaled parameters (batch, 20)
        y_spectra: target spectra (batch, 1000)
        lambda_physics: weight for physics loss
        physics_subsample: number of samples to run through TMM per batch
                          (TMM is expensive, so we subsample)
    """
    # ── Part 1: Standard NLL Loss (full batch) ──
    z, log_det_J = model(x_params, y_spectra)
    loss_nll = torch.mean(0.5 * torch.sum(z**2, dim=1) - log_det_J)
    
    # ── Part 2: Physics Loss (subsampled) ──
    if lambda_physics > 0:
        # Subsample to keep TMM cost manageable
        idx = torch.randperm(x_params.size(0))[:physics_subsample]
        y_sub = y_spectra[idx]
        
        # Generate parameters via reverse pass
        with torch.no_grad():
            z_sample = torch.randn(physics_subsample, 20, device=device)
        
        # This part MUST be differentiable — don't use torch.no_grad()
        x_generated = model.reverse(z_sample, y_sub)
        
        # Run through differentiable TMM
        spectra_reconstructed = differentiable_tmm(x_generated, scaler_params, device)
        
        # MSE between target and reconstructed spectra
        loss_physics = F.mse_loss(spectra_reconstructed, y_sub)
    else:
        loss_physics = torch.tensor(0.0, device=device)
    
    loss_total = loss_nll + lambda_physics * loss_physics
    
    return loss_total, loss_nll.item(), loss_physics.item()



### Step 4: Updated Training Loop



In [ ]:
# Hyperparameters for physics loss
LAMBDA_PHYSICS = 0.1          # Start small, increase if needed
PHYSICS_SUBSAMPLE = 64        # TMM on 64 samples per batch (vs 2048 full batch)
WARMUP_EPOCHS = 20            # Train pure NLL first, then add physics loss

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=250)

for epoch in range(250):
    model.train()
    
    # Warm up: pure NLL for first 20 epochs, then add physics
    current_lambda = LAMBDA_PHYSICS if epoch >= WARMUP_EPOCHS else 0.0
    
    epoch_nll = 0
    epoch_phys = 0
    
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        loss_total, nll_val, phys_val = combined_loss(
            model, x_batch, y_batch, scaler_params, device,
            lambda_physics=current_lambda,
            physics_subsample=PHYSICS_SUBSAMPLE
        )
        
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_nll += nll_val
        epoch_phys += phys_val
    
    scheduler.step()
    
    num_batches = len(train_loader)
    print(f"Epoch {epoch+1}: NLL={epoch_nll/num_batches:.4f}, "
          f"Physics={epoch_phys/num_batches:.6f}, "
          f"λ_phys={current_lambda}")



### Step 5: Gradient Flow Diagram with Physics Loss



In [ ]:
┌─────────────────────────────────────────────────────────────────────────┐
│                  TRAINING WITH PHYSICS LOSS                             │
│                                                                         │
│  PATH 1 (NLL Loss):                                                     │
│                                                                         │
│    x_params ──→ [cINN Forward] ──→ z, log_det_J                        │
│    y_spectra ─→ [Encoder] ──────→ cond ──┘          │                   │
│                                                      ▼                  │
│                                                  L_NLL = 0.5||z||²      │
│                                                        - log|det(J)|    │
│                                                                         │
│  PATH 2 (Physics Loss):                                                 │
│                                                                         │
│    z ~ N(0,I) ──→ [cINN Reverse] ──→ x_generated ──→ [Differentiable]  │
│    y_spectra ──→ [Encoder] ──────→ cond ──┘            │   TMM          │
│                                                        ▼                │
│                                                  spectra_pred           │
│                                                        │                │
│                                                        ▼                │
│                                             L_physics = MSE(spectra_pred│
│                                                        , y_spectra)     │
│                                                                         │
│  COMBINED:                                                              │
│                                                                         │
│    L_total = L_NLL + λ * L_physics                                      │
│        │                                                                │
│        ▼                                                                │
│    backprop updates:                                                    │
│      • All 8 SubNets (via both paths)                                   │
│      • SpectrumEncoder (via both paths)                                 │
│                                                                         │
│  GRADIENT FLOW:                                                         │
│                                                                         │
│    L_physics → ∂L/∂spectra_pred → ∂TMM/∂params → ∂reverse/∂SubNet_W   │
│                                                  → ∂reverse/∂Encoder_W  │
│                                                                         │
│    L_NLL    → ∂L/∂z → ∂forward/∂SubNet_W                               │
│             → ∂L/∂log_det → ∂forward/∂SubNet_W                         │
│                            → via cond → ∂forward/∂Encoder_W            │
└─────────────────────────────────────────────────────────────────────────┘



### Important Considerations

| Issue | Solution |
|-------|----------|
| **TMM is expensive** | Subsample: run physics loss on 64/2048 = 3% of batch |
| **TMM may have numerical instabilities** | `soft_clamp` instead of `clamp`; add `+ 1e-10` inside `sqrt` and `abs` in TMM |
| **Physics loss can dominate early** | Warmup: 20 epochs of pure NLL first |
| **λ tuning** | Start with 0.01–0.1; if physics loss is ~100x larger than NLL, reduce λ |
| **`z_sample` not differentiable** | This is correct — `z` is a random input, not a learnable parameter. Gradients flow through the reverse network, not through `z` |
| **Reverse pass must be differentiable** | Don't wrap `model.reverse()` in `torch.no_grad()` for the physics path |

### Potential Issue in Your TMM Code

Check your TMM for these non-differentiable operations:



In [ ]:
# ❌ These break gradients:
np.abs(...)          # Use torch.abs instead
.cpu().numpy()       # Stay in torch
torch.clamp(...)     # Use soft_clamp (zero grad at boundary)

# ✅ These are fine:
torch.abs(...)
torch.sqrt(... + 1e-10)   # Add epsilon for numerical stability
torch.exp(...)
torch.matmul(...)



Your `calculate_absorption_full_curve_gpu` already uses pure torch operations, so it should be mostly differentiable. The only thing to fix is replacing `torch.clamp` with `soft_clamp` and ensuring no detach/numpy calls exist in the path.

### Expected Impact

| Metric | Without Physics Loss | With Physics Loss (expected) |
|--------|---------------------|------------------------------|
| OOB Rate | Current value | Slightly lower |
| Best candidate RMSE | Current value | 20–50% improvement |
| Hit rate (RMSE < 0.05) | Current value | Higher |
| Training time per epoch | ~X seconds | ~1.3–1.5X seconds |
| Latent space Gaussianity | Good | Slightly worse (tradeoff) |

The physics loss forces the network to produce parameters that **actually work** when simulated, rather than just making the latent space Gaussian. The tradeoff is that the latent space may become slightly non-Gaussian, but the practical design quality improves.